# 04 · QA Pairs — RAG Evaluation Ground Truth
20 curated question/answer pairs with `policy_ref` and `suggested_action` — the only ground truth available for evaluating retrieval + action-suggestion quality end-to-end.

In [1]:
import sys
sys.path.append('../src')
import pandas as pd
from data_loader import load_qa_pairs
pd.set_option('display.max_colwidth', 100)
qa = load_qa_pairs()
qa.head()

,id,category,question,answer,policy_ref,risk_level,suggested_action
0,QA001,Fraud,"I noticed an unauthorized transaction of ₹15,000 on my account. What should I do?","Please report immediately via our 24x7 helpline or mobile app. We will block your card, raise a ...","Fraud Handling Policy §3, §4",High,"Block card, raise dispute, escalate to fraud team"
1,QA002,Fraud,How long does fraud investigation take?,Fraud investigations are completed within 30-45 working days. You will receive updates every 7 d...,Fraud Handling Policy §4,Medium,Update customer with reference number
2,QA003,Fraud,Someone used my OTP to transfer money. Can I get a refund?,"If the OTP was shared voluntarily, liability may apply per RBI guidelines. If it was obtained th...","Fraud Handling Policy §3.2, §6",High,"Suspend account access, escalate to fraud team, advise FIR"
3,QA004,Fraud,Multiple small transactions are showing up that I didn't make,Multiple micro-transactions could indicate card skimming or credential compromise. Your card wil...,Fraud Handling Policy §5 High Risk,High,"Block card, dispatch new card, file FIR"
4,QA005,Fraud,I received an OTP I didn't request. Is my account safe?,An unrequested OTP could indicate a fraud attempt or SIM swap. Your account has been placed unde...,"Fraud Handling Policy §5, §6.2",High,"Suspend account, advise branch visit, reset credentials"


## 1. Category / risk coverage
Only 20 examples total, unevenly split across category and risk_level — too small to be a statistically reliable benchmark on its own, but still useful as a spot-check and as the seed for `action_layer.py`'s (category, risk_level) -> action table.

In [2]:
print(qa['category'].value_counts())
print()
print(qa['risk_level'].value_counts())

category
Fraud             6
Loan              6
KYC               5
Account Access    3
Name: count, dtype: int64

risk_level
Low       11
High       6
Medium     3
Name: count, dtype: int64


## 2. Coverage gaps for the action table
`action_layer.py` derives one action per (category, risk_level) pair straight from this data. With only 20 rows across 4 categories x 3 risk levels (12 possible combinations), several combinations have no example at all — those fall back to a generic 'route to human agent' action. Worth expanding qa_pairs.json if this table needs full coverage.

In [3]:
combos = qa.groupby(['category', 'risk_level']).size().unstack(fill_value=0)
combos

risk_level,High,Low,Medium
category,,,
Account Access,0,3,0
Fraud,5,0,1
KYC,0,3,2
Loan,1,5,0


## 3. Answer length — useful as a target length for generated responses

In [4]:
qa['answer'].str.split().apply(len).describe()

count    20.000000
mean     38.200000
std       5.136044
min      28.000000
25%      34.750000
50%      39.500000
75%      42.000000
max      46.000000
Name: answer, dtype: float64

## 4. Takeaways
- 20 examples is enough to sanity-check retrieval and seed the action table, not enough to be a statistically confident benchmark — treat retrieval/action metrics as directional.
- Several (category, risk_level) combinations have no seed example — those actions fall back to a generic default; expanding this file would make the action layer fully data-driven instead of partially defaulted.
- Reference answers average a specific, actionable length (~30-40 words) — useful as a style/length target if generation output is ever compared against these.